# Stage 2: GPU Training (Run on Kaggle 2x T4)

This notebook contains **zero** image generation logic. It is purely designed to train the ensemble models on Kaggle GPUs as fast as possible.

**Prerequisites:**
You must have uploaded the zipped `./kaggle_dataset` directory from the Stage 1 notebook as a Kaggle Dataset and attached it to this notebook.

**Architecture:**
- Uses the `MemmapDataset` to stream raw bytes directly from the read-only `/kaggle/input/...` directory into the GPUs via DMA, bypassing Kaggle's disk-write quota.
- `MAX_PARALLEL = 4` trains 4 models concurrently to max out the 2x T4 GPUs.


In [ ]:
import os, tempfile, time
import concurrent.futures as cf
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

HAS_CUDA = torch.cuda.is_available()
DEVICES  = [f'cuda:{i}' for i in range(torch.cuda.device_count())] if HAS_CUDA else ['cpu']

# ---- knobs ----
BATCH_SIZE   = 128
EPOCHS       = 100
LR           = 1e-5
PATIENCE     = 2
NUM_WORKERS  = 2  
USE_AMP      = False 
MAX_PARALLEL = len(DEVICES) * 2  # 4 concurrent models on 2x T4

# --- ensemble diversity ---
BAGGING           = True
TRAIN_FRAC        = 0.7
SHARED_SPLIT_SEED = 42

# ⚠️ UPDATE THIS PATH to point to your attached Kaggle Dataset
KAGGLE_INPUT_DIR = '/kaggle/input/trend-images' 

print(f"Devices: {DEVICES} | Max Parallel: {MAX_PARALLEL} | Dataset: {KAGGLE_INPUT_DIR}")


## 1. Load Market Data (for alignment)


In [ ]:
# Load original market data to get the forward returns aligned
# You must also upload the original parquet file to Kaggle!
file_name = '/kaggle/input/yashkatiyar409/market-data-sp500-1997-2023-parquet'
if not os.path.exists(file_name):
    # fallback if you put it in the same dataset
    file_name = os.path.join(KAGGLE_INPUT_DIR, 'market_data_sp500_1997_2023.parquet')

df_market = pd.read_parquet(file_name)
print(f"Loaded {len(df_market):,} rows of market data.")


## 2. Horizons, Splits, and DMA Dataset


In [ ]:
def _height(n_days): return 32 if n_days==5 else (64 if n_days==20 else 96)

def label_for_horizon(n_days, horizon):
    meta_path = os.path.join(KAGGLE_INPUT_DIR, f'meta_n{n_days}.parquet')
    meta_df = pd.read_parquet(meta_path)
    
    # We still need the original dataframe grouped by ticker to extract forward returns
    grouped = {t: df_market[df_market['Ticker']==t].copy().sort_values('Date').reset_index(drop=True) 
               for t in meta_df['ticker'].unique()}
    
    # Calculate forward returns dynamically
    fwd_rets = np.full(len(meta_df), np.nan)
    for t in meta_df['ticker'].unique():
        t_mask = meta_df['ticker'] == t
        t_meta = meta_df[t_mask]
        t_df = grouped[t]
        
        ei = t_meta['end_idx'].values
        valid_fwd = (ei + horizon) < len(t_df)
        
        base_prices = t_df['Close'].values[ei]
        fwd_prices = np.where(valid_fwd, t_df['Close'].values[np.minimum(ei+horizon, len(t_df)-1)], np.nan)
        
        fwd_rets[t_mask] = (fwd_prices / base_prices) - 1.0
        
    meta_df['Fwd_Ret'] = fwd_rets
    valid = ~np.isnan(meta_df['Fwd_Ret'])
    
    rows = np.where(valid)[0]
    label = (meta_df['Fwd_Ret'].values[rows] > 0).astype(np.int64)
    base = meta_df.iloc[rows].copy().rename(columns={'ticker': 'Ticker', 'end_date': 'Date'})
    
    return rows, label, base, len(meta_df)

def trainval_positions(base_df):
    d = base_df['Date'].values
    return np.where((d>='1997-01-01') & (d<='2004-12-31'))[0]

def test_positions(base_df):
    d = base_df['Date'].values
    te = np.where((d>='2005-01-01') & (d<='2023-12-31'))[0]
    return te[np.argsort(d[te], kind='stable')]

def subsample_split(tv_positions, seed, train_frac=0.7):
    tv = tv_positions.copy()
    np.random.default_rng(seed).shuffle(tv)
    cut = int(train_frac*len(tv))
    return tv[:cut], tv[cut:]

class MemmapDataset(Dataset):
    def __init__(self, dat_path, shape, image_rows, labels=None):
        self.dat_path = dat_path
        self.shape = shape
        self.rows = np.asarray(image_rows)
        self.labels = labels
        self._mm = None
        
    def __len__(self): return len(self.rows)
    
    def __getitem__(self, i):
        if self._mm is None: 
            self._mm = np.memmap(self.dat_path, dtype=np.uint8, mode='r', shape=self.shape)
        
        # OS DMA read directly from read-only Kaggle Dataset to RAM
        img = torch.from_numpy(np.ascontiguousarray(self._mm[int(self.rows[i])])) 
        if self.labels is None: return img
        return img, int(self.labels[i])


## 3. CNN architecture


In [ ]:
def xavier_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.zeros_(m.bias)

class TrendCNN(nn.Module):
    def __init__(self, n_days=20):
        super().__init__()
        if n_days == 5:    blocks, v_stride, v_dilation = 2, 1, 1
        elif n_days == 20: blocks, v_stride, v_dilation = 3, 3, 2
        else:              blocks, v_stride, v_dilation = 4, 3, 3
        layers, in_c, out_c = [], 1, 64
        for i in range(blocks):
            stride   = (v_stride, 1) if i == 0 else (1, 1)
            dilation = (v_dilation, 1) if i == 0 else (1, 1)
            layers += [nn.Conv2d(in_c, out_c, (5,3), stride=stride, padding=(2,1), dilation=dilation),
                       nn.BatchNorm2d(out_c), nn.LeakyReLU(0.01), nn.MaxPool2d((2,1))]
            in_c, out_c = out_c, out_c*2
        self.features = nn.Sequential(*layers)
        t = torch.zeros(1,1,_height(n_days), n_days*3)
        flat = self.features(t).view(1,-1).size(1)
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(flat, 2))
        self.apply(xavier_init)
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.view(x.size(0), -1))


## 4. Training engine


In [ ]:
def _mk_loader(ds, shuffle, device):
    kw = dict(batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=('cuda' in device))
    if NUM_WORKERS > 0: kw.update(persistent_workers=True, prefetch_factor=4)
    return DataLoader(ds, **kw)

def train_model(model, train_loader, val_loader, dev):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    amp = USE_AMP and dev.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp)
    best, pc = float('inf'), 0
    ckpt = tempfile.NamedTemporaryFile(suffix='.pt', delete=False).name
    
    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train(); train_loss = 0.0
        for imgs, labels in train_loader:
            imgs   = imgs.to(dev, non_blocking=True).float().div_(255.0)
            labels = labels.to(dev, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if amp:
                with torch.autocast('cuda'): loss = criterion(model(imgs), labels)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                loss = criterion(model(imgs), labels)
                loss.backward(); optimizer.step()
            train_loss += loss.item()
        train_loss /= max(1, len(train_loader))
                
        model.eval(); val_loss = 0.0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs   = imgs.to(dev, non_blocking=True).float().div_(255.0)
                labels = labels.to(dev, non_blocking=True)
                if amp:
                    with torch.autocast('cuda'): val_loss += criterion(model(imgs), labels).item()
                else: val_loss += criterion(model(imgs), labels).item()
        val_loss /= max(1, len(val_loader))
        
        elapsed = time.time() - t0
        if val_loss < best:
            best, pc = val_loss, 0
            torch.save(model.state_dict(), ckpt)
            print(f"      Epoch {epoch+1:02d} | T-Loss: {train_loss:.4f} | V-Loss: {val_loss:.4f} | Time: {elapsed:.1f}s | [BEST]")
        else:
            pc += 1
            print(f"      Epoch {epoch+1:02d} | T-Loss: {train_loss:.4f} | V-Loss: {val_loss:.4f} | Time: {elapsed:.1f}s | Patience: {pc}/{PATIENCE}")
            if pc >= PATIENCE:
                print(f"      Early stopping triggered after {epoch+1} epochs.")
                break
            
    model.load_state_dict(torch.load(ckpt, map_location=dev, weights_only=True))
    os.remove(ckpt)
    return model

def predict_probs(model, loader, dev):
    model.eval(); out=[]
    with torch.no_grad():
        for imgs in loader:
            imgs = imgs.to(dev, non_blocking=True).float().div_(255.0)
            if USE_AMP and dev.type=='cuda':
                with torch.autocast('cuda'): p = torch.softmax(model(imgs), 1)[:, 1]
            else: p = torch.softmax(model(imgs), 1)[:, 1]
            out.append(p.float().cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def train_and_predict(job):
    n_days, total_N, tr_rows, tr_lab, va_rows, va_lab, te_rows, device, seed = job
    torch.manual_seed(seed)
    dev = torch.device(device)
    if dev.type == 'cuda': torch.cuda.set_device(dev)
    
    dat_path = os.path.join(KAGGLE_INPUT_DIR, f'raw_n{n_days}.dat')
    shape = (total_N, 1, _height(n_days), n_days*3)
    
    tl = _mk_loader(MemmapDataset(dat_path, shape, tr_rows, tr_lab), True, device)
    vl = _mk_loader(MemmapDataset(dat_path, shape, va_rows, va_lab), False, device)
    
    model = train_model(TrendCNN(n_days).to(dev), tl, vl, dev)
    te_l = _mk_loader(MemmapDataset(dat_path, shape, te_rows), False, device)
    return predict_probs(model, te_l, dev)


## 5. Experiment loop


In [ ]:
IMAGE_SIZES, HORIZONS, ENSEMBLE_SIZE = [5, 20, 60], [5, 20, 60], 5

def calc_sharpe(signal_col, df, horizon):
    try:
        # STRICT PAPER EVALUATION: Enforce non-overlapping returns.
        dates = np.sort(df['Date'].unique())
        non_overlap_dates = dates[::horizon] 
        df_sub = df[df['Date'].isin(non_overlap_dates)].copy()
        
        df_sub['Decile'] = df_sub.groupby('Date')[signal_col].transform(
            lambda x: pd.qcut(x.rank(method='first'), 10, labels=False))
            
        high = df_sub[df_sub['Decile']==9].groupby('Date')['Fwd_Ret'].mean()
        low  = df_sub[df_sub['Decile']==0].groupby('Date')['Fwd_Ret'].mean()
        spread = high - low
        
        periods_per_year = 252 / horizon
        annual_spread = spread.mean() * periods_per_year
        annual_vol    = spread.std() * np.sqrt(periods_per_year)
        
        return annual_spread, (annual_spread/annual_vol if annual_vol != 0 else 0)
    except Exception:
        return 0, 0

results = []
for n_days in IMAGE_SIZES:
    for horizon in HORIZONS:
        print(f"\n{'='*54}\nConfig I{n_days} / R{horizon}\n{'='*54}")
        rows, label, base, total_N = label_for_horizon(n_days, horizon)
        te     = test_positions(base)
        tv_all = trainval_positions(base)
        if len(tv_all)==0 or len(te)==0: continue
            
        te_rows   = rows[te]
        eval_base = base.iloc[te].reset_index(drop=True)

        jobs = []
        for i in range(ENSEMBLE_SIZE):
            seed = 1000 + i
            split_seed = seed if BAGGING else SHARED_SPLIT_SEED
            tr, va = subsample_split(tv_all, split_seed, TRAIN_FRAC)
            jobs.append((n_days, total_N, rows[tr], label[tr], rows[va], label[va], te_rows, DEVICES[i % len(DEVICES)], seed))

        preds = [None]*ENSEMBLE_SIZE
        if MAX_PARALLEL > 1 and DEVICES[0] != 'cpu':
            print(f"  training {ENSEMBLE_SIZE} models, {MAX_PARALLEL} at a time across {len(DEVICES)} GPU(s)...")
            with cf.ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
                fut = {ex.submit(train_and_predict, j): k for k, j in enumerate(jobs)}
                for f in tqdm(cf.as_completed(fut), total=len(fut), desc="  ensemble", leave=False):
                    preds[fut[f]] = f.result()
        else:
            for k, j in enumerate(jobs):
                print(f"  --- model {k+1}/{ENSEMBLE_SIZE} on {j[7]} ---")
                preds[k] = train_and_predict(j)

        eval_base['CNN_Prob'] = np.mean(preds, axis=0)
        for strat in ['CNN_Prob','MOM','STR','WSTR','TREND']:
            ret, sharpe = calc_sharpe(strat, eval_base.copy(), horizon)
            results.append({'Config':f"I{n_days}/R{horizon}", 'Strategy':strat, 'Ann_Return':ret, 'Ann_Sharpe':sharpe})
            
df_results = pd.DataFrame(results)
display(df_results)
df_results.to_csv("final_experiment_results_kaggle.csv", index=False)
print("\nDone. Results -> final_experiment_results_kaggle.csv")
